# Week 2 · Module 2 Practical — The Meaning Machine 🧭

**PMS RoBoTics Research Center · 4-Week Internship & Training Program**
**Week 2 · Module 2: Text embeddings and how machines understand meaning (+ context-aware chunking)**

---

### What you'll do in the next 30 minutes

| # | Experiment | Skill |
|---|-----------|-------|
| 1 | Touch the vectors | Embeddings API, cosine similarity by hand |
| 2 | Semantic retriever | Swap BM25 → embeddings, same interface |
| 3 | Head-to-head eval | **BM25 vs embeddings on the SAME gold set** (P@3, R@3, F1, MRR) |
| 4 | Chunking lab | One messy policy doc · 3 strategies · measured winner |
| 5 | Where vectors lose | The X-500 test — why hybrid exists |
| 🏁 | **RAGBot v0.2** | Semantic retrieval + CA-chunked policies + before/after scorecard |

> 📏 **The habit:** change one component → re-run the same eval → compare numbers.

## Step 0 — Setup 🔑

In [ ]:
# Setup — install only what we need; leave Colab's numpy alone
%pip install -q -U openai rank_bm25

In [ ]:
from getpass import getpass
from openai import OpenAI
import numpy as np

API_KEY = getpass("Paste the OpenAI API key: ")
client = OpenAI(api_key=API_KEY)
CHAT_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"   # 1,536 dims, ~$0.02 per 1M tokens

def ask(prompt, temperature=0.3, max_tokens=250):
    r = client.chat.completions.create(model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content.strip()

def embed_batch(texts):
    """Embed a list of texts in one API call. Returns an (n, 1536) numpy array."""
    r = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in r.data])

print("✅ Ready:", CHAT_MODEL, "+", EMBED_MODEL)

In [ ]:
# Yesterday's knowledge base — unchanged (needed for the head-to-head)
CATALOG = [
    "P-101 | Prima Electric Kettle 1.5L | Rs. 1,299 | kitchen | 1500W fast boil, auto shut-off, 1-year warranty | in stock: 42",
    "P-102 | Prima Electric Kettle 2.0L | Rs. 1,699 | kitchen | family size, keep-warm mode, 1-year warranty | in stock: 18",
    "P-103 | Zenith Steel Kettle 1.0L | Rs. 899 | kitchen | budget stovetop kettle, no electricity needed | in stock: 65",
    "P-104 | Zenith Pro Kettle 1.8L | Rs. 2,499 | kitchen | premium, temperature control, 2-year warranty | in stock: 7",
    "P-105 | SwiftMix Mixer Grinder 750W | Rs. 3,499 | kitchen | 3 jars, overload protection, 2-year warranty | in stock: 23",
    "P-106 | SwiftMix Mixer Grinder 500W | Rs. 2,299 | kitchen | 2 jars, compact, 1-year warranty | in stock: 31",
    "P-107 | AirChef Air Fryer 4L | Rs. 5,999 | kitchen | oil-free frying, digital timer, 1-year warranty | in stock: 12",
    "P-108 | AirChef Air Fryer 6L XL | Rs. 8,499 | kitchen | family size, 8 presets, 2-year warranty | in stock: 4",
    "P-109 | CoolBreeze Table Fan 400mm | Rs. 1,899 | appliances | 3-speed, oscillating, 2-year warranty | in stock: 55",
    "P-110 | CoolBreeze Tower Fan | Rs. 4,299 | appliances | remote control, timer, quiet mode | in stock: 9",
    "P-111 | BrightHome LED Bulb 9W (pack of 4) | Rs. 499 | electrical | warm white, 2-year replacement | in stock: 210",
    "P-112 | BrightHome Smart Bulb WiFi | Rs. 799 | electrical | app control, 16M colors, voice assistant support | in stock: 48",
    "P-113 | PowerSafe Extension Board 6-socket | Rs. 649 | electrical | surge protection, 2m cord | in stock: 87",
    "P-114 | SmartWatch Fit Pro | Rs. 2,999 | gadgets | heart rate, SpO2, 7-day battery | in stock: 26",
    "P-115 | EarBuds Bass+ TWS | Rs. 1,499 | gadgets | 24h playback, IPX4, ENC mic | in stock: 39",
    "P-116 | PixelSnap Barcode Scanner X-500 | Rs. 4,999 | business | wireless, 18h battery, 1-year warranty | in stock: 6",
    "P-117 | ThermoSteel Water Bottle 1L | Rs. 749 | home | 12h hot / 24h cold, leak-proof | in stock: 120",
    "P-118 | CleanSweep Robot Vacuum | Rs. 12,999 | home | app control, auto-dock, HEPA filter | in stock: 3",
    "P-119 | FreshBrew Coffee Maker 600ml | Rs. 2,799 | kitchen | drip brew, keep-warm plate, 1-year warranty | in stock: 14",
    "P-120 | ChopMaster Vegetable Chopper | Rs. 599 | kitchen | 900ml, 3 blades, dishwasher safe | in stock: 73",
    "D-201 | POLICY delivery | Free delivery above Rs. 999, otherwise Rs. 49. Standard 2-4 days in Pune, 4-7 days rest of Maharashtra.",
    "D-202 | POLICY returns | Returns within 7 days with receipt for full refund. Without receipt: store credit only. Electronics must be unopened.",
    "D-203 | POLICY warranty | Warranty claims need invoice + product serial number. Service center: SmartMart Pimple Saudagar, 9 AM-9 PM.",
    "D-204 | POLICY offers | Current offer: 10% off all kitchen appliances till Sunday. Not combinable with other coupons.",
]
print(f"{len(CATALOG)} chunks loaded")

---
## Experiment 1 — Touch the vectors 🖐️

No abstractions. Embed real sentences, print the numbers, measure similarity yourself:

In [ ]:
sentences = [
    "electric kettle for boiling water",     # 0
    "sasta chai banane ka bartan",           # 1 — Hinglish, zero shared words with 0
    "robot vacuum cleaner for floors",       # 2
    "wireless earbuds with good bass",       # 3
]
vecs = embed_batch(sentences)
print("Shape:", vecs.shape)                  # (4, 1536)
print("First 8 numbers of sentence 0:", np.round(vecs[0][:8], 3))

In [ ]:
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("Similarity matrix (1.0 = identical meaning):\n")
print(" " * 4 + "".join(f"  s{j}  " for j in range(len(sentences))))
for i in range(len(sentences)):
    row = "".join(f" {cosine(vecs[i], vecs[j]):.2f} " for j in range(len(sentences)))
    print(f"s{i} {row}   {sentences[i][:38]}")

**Read the matrix:** s0 (kettle) vs s1 (Hinglish chai bartan) should be the highest off-diagonal pair — *similar meaning, zero shared words, different languages*. s0 vs s2 (vacuum) much lower. That one comparison is the entire reason today's module exists.

**✏️ Your turn:** add a 5th sentence — your own paraphrase of s0 in any language — and re-run. Where does it land?

---
## Experiment 2 — The semantic retriever 🔎

Same `retrieve(query, k)` interface as yesterday's BM25 — so the RAG loop and eval need **zero changes**. That's deliberate design:

In [ ]:
# Index time: embed all chunks ONCE (one batched call), normalise to unit length
chunk_vecs = embed_batch(CATALOG)
chunk_vecs = chunk_vecs / np.linalg.norm(chunk_vecs, axis=1, keepdims=True)
print(f"Indexed {chunk_vecs.shape[0]} chunks x {chunk_vecs.shape[1]} dims")

def retrieve_semantic(query, k=3):
    q = embed_batch([query])[0]
    q = q / np.linalg.norm(q)
    scores = chunk_vecs @ q                       # dot of unit vectors = cosine
    ranked = np.argsort(scores)[::-1][:k]
    return [(CATALOG[i], float(scores[i])) for i in ranked]

# Yesterday's BM25 failures — the moment of truth:
for q in ["something affordable for boiling water",
          "sasta kettle dikhao",
          "birthday gift under 1500 rupees"]:
    print(f"Q: {q}")
    for chunk, score in retrieve_semantic(q, k=2):
        print(f"   [{score:.3f}] {chunk[:70]}...")
    print()

**The queries that scored 0.00 on BM25 now land on the right chunks** — including the Hinglish one. No translation step, no synonym lists. The meaning geometry does it all.

**✏️ Your turn:** try your own weirdest paraphrase — "machine that makes hot wind for frying without oil"?

---
## Experiment 3 — Head-to-head, same gold set ⚔️

Yesterday's TEST_SET, **extended with the synonym/Hinglish queries from homework**. Both retrievers, all four metrics, one table:

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return text.lower().replace("|", " ").replace(",", " ").split()

bm25 = BM25Okapi([tokenize(c) for c in CATALOG])

def retrieve_bm25(query, k=3):
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(range(len(CATALOG)), key=lambda i: scores[i], reverse=True)[:k]
    return [(CATALOG[i], scores[i]) for i in ranked]

TEST_SET = [
    # yesterday's 10
    ("price of prima 1.5L kettle",              {"P-101"}),
    ("cheapest kettle",                          {"P-103"}),
    ("robot vacuum in stock?",                   {"P-118"}),
    ("X-500 scanner warranty",                   {"P-116", "D-203"}),
    ("return policy without receipt",            {"D-202"}),
    ("delivery charge for Rs. 500 order",        {"D-201"}),
    ("discount on kitchen items",                {"D-204"}),
    ("mixer grinder with overload protection",   {"P-105"}),
    ("affordable way to boil water",             {"P-103", "P-101"}),
    ("smart lighting for home",                  {"P-112"}),
    # homework extension: BM25's blind spots
    ("sasta kettle dikhao",                      {"P-103"}),
    ("something to make coffee in the morning",  {"P-119"}),
    ("gift idea: music on the go",               {"P-115"}),
]

def chunk_id(chunk):
    return chunk.split("|")[0].strip().split()[0]

def evaluate(retriever, k=3, test_set=TEST_SET, quiet=True):
    ps, rs, f1s, rrs = [], [], [], []
    for query, relevant in test_set:
        retrieved_ids = [chunk_id(c) for c, _ in retriever(query, k=k)]
        hits = [rid for rid in retrieved_ids if rid in relevant]
        p = len(hits) / k
        r = len(set(hits)) / len(relevant)
        f1 = 2 * p * r / (p + r) if (p + r) else 0.0
        rr = next((1.0 / rank for rank, rid in enumerate(retrieved_ids, 1) if rid in relevant), 0.0)
        ps.append(p); rs.append(r); f1s.append(f1); rrs.append(rr)
        if not quiet:
            print(f"  {query:<44} P {p:.2f}  R {r:.2f}  RR {rr:.2f}")
    n = len(test_set)
    return {"P@k": sum(ps)/n, "R@k": sum(rs)/n, "F1": sum(f1s)/n, "MRR": sum(rrs)/n}

print(f"{'metric':<8} {'BM25':>8} {'Embeddings':>12}")
bm = evaluate(retrieve_bm25, k=3)
em = evaluate(retrieve_semantic, k=3)
for m in ["P@k", "R@k", "F1", "MRR"]:
    delta = em[m] - bm[m]
    print(f"{m:<8} {bm[m]:>8.2f} {em[m]:>12.2f}   ({'+' if delta>=0 else ''}{delta:.2f})")

**One component changed, same eval, numbers compared.** Embeddings should win overall — driven almost entirely by the synonym/Hinglish queries recovering from zero.

**Look closer though:** run `evaluate(retrieve_semantic, k=3, quiet=False)` and find queries where embeddings did *worse* than BM25. Note them — Experiment 5 explains why.

---
## Experiment 4 — The chunking lab ✂️

A realistic mess: SmartMart's **combined policy & FAQ document** — one wall of text, exactly how businesses actually hand you their documents. We chunk it **three ways** and measure which one answers questions best.

In [ ]:
POLICY_DOC = """SmartMart Customer Service Handbook

Section 1: Returns and Refunds
Products can be returned within 7 days of purchase for a full refund, provided the customer presents the original receipt. If the receipt is lost, we offer store credit equal to the current selling price. Electronics must be unopened with seals intact. It must also be in the original packaging with all accessories included. Kitchen appliances that have been used can only be returned if defective. Defective items are eligible for free replacement within 15 days instead of refund if the customer prefers.

Section 2: Warranty Claims
All warranty claims require the original invoice and the product serial number. The standard process takes 7-10 working days. Premium members get pickup and drop service for warranty repairs at no extra charge. For products above Rs. 5,000, we offer doorstep inspection before pickup. The warranty does not cover physical damage, water damage, or unauthorized repairs. It becomes void if the product serial number is tampered with.

Section 3: EMI and Payment Options
Purchases above Rs. 3,000 are eligible for EMI on all major credit cards. We support 3, 6, 9 and 12 month tenures. No-cost EMI is available on select kitchen appliances during festival sales. UPI, cards, cash and store gift vouchers are accepted at all counters. Gift vouchers cannot be combined with no-cost EMI offers.

Section 4: SmartMart Premium Membership
Premium membership costs Rs. 499 per year. Benefits include free delivery on ALL orders regardless of value, early access to festival sales, an extra 5% discount on Tuesdays, and free pickup for warranty repairs. Membership can be cancelled within 30 days for a full refund if no benefits have been used."""

print(f"Document: {len(POLICY_DOC)} chars, {len(POLICY_DOC.split())} words")

In [ ]:
# ---- Strategy A: fixed-size (naive) — cut every 300 characters, no mercy ----
def chunk_fixed(doc, size=300):
    return [doc[i:i+size] for i in range(0, len(doc), size)]

# ---- Strategy B: structural — split on paragraphs (blank lines) ----
def chunk_structural(doc):
    return [p.strip() for p in doc.split("\n\n") if p.strip()]

# ---- Strategy C: context-aware — structural + section header injected + overlap ----
def chunk_context_aware(doc, overlap_sentences=1):
    import re
    title = doc.split("\n")[0].strip()
    sections = re.split(r"\n(?=Section \d+:)", doc)
    chunks = []
    for sec in sections[1:]:                       # skip the title block
        lines = sec.strip().split("\n")
        header = lines[0].strip()                  # e.g. "Section 1: Returns and Refunds"
        body = " ".join(lines[1:]).strip()
        sentences = re.split(r"(?<=[.!?]) +", body)
        # group ~3 sentences per chunk with 1-sentence overlap
        step = 3 - overlap_sentences
        for i in range(0, len(sentences), step):
            group = sentences[i:i+3]
            if not group: continue
            chunks.append(f"[{title} > {header}] " + " ".join(group))
            if i + 3 >= len(sentences): break
    return chunks

for name, fn in [("A fixed-size", chunk_fixed), ("B structural", chunk_structural), ("C context-aware", chunk_context_aware)]:
    ch = fn(POLICY_DOC)
    print(f"Strategy {name}: {len(ch)} chunks")
print()
print("--- sample chunk from each ---")
print("A:", chunk_fixed(POLICY_DOC)[3][:120], "...")
print()
print("B:", chunk_structural(POLICY_DOC)[2][:120], "...")
print()
print("C:", chunk_context_aware(POLICY_DOC)[6][:160], "...")

**Look at sample A** — it almost certainly cuts mid-sentence, and you can't tell which section it's from. **Sample C** carries `[Handbook > Section N]` context in every chunk. Now let's *measure* instead of admire:

In [ ]:
POLICY_TESTS = [
    ("can I get a refund without my receipt?",            "store credit"),
    ("do used kitchen appliances qualify for return?",    "defective"),
    ("how long does a warranty claim take?",              "7-10"),
    ("is there doorstep inspection for expensive items?", "5,000"),
    ("what EMI tenures do you support?",                  "12"),
    ("what do premium members get on Tuesdays?",          "5%"),
    ("can I cancel my membership?",                       "30 days"),
]

def eval_chunking(chunks, tests=POLICY_TESTS, k=2):
    """Retrieval hit-rate: does the expected keyphrase appear in the top-k retrieved text?"""
    cv = embed_batch(chunks)
    cv = cv / np.linalg.norm(cv, axis=1, keepdims=True)
    hits = 0
    for query, must_contain in tests:
        q = embed_batch([query])[0]; q = q / np.linalg.norm(q)
        top = [chunks[i] for i in np.argsort(cv @ q)[::-1][:k]]
        hits += any(must_contain.lower() in t.lower() for t in top)
    return hits / len(tests)

print("Chunking strategy      hit-rate@2")
for name, fn in [("A fixed-size     ", chunk_fixed),
                 ("B structural     ", chunk_structural),
                 ("C context-aware  ", chunk_context_aware)]:
    rate = eval_chunking(fn(POLICY_DOC))
    print(f"{name}  {rate:.0%}")

**Typical result: C ≥ B > A.** Fixed-size loses answers at chunk boundaries; structural is decent (this doc has clean paragraphs — real PDFs rarely do); context-aware wins because every chunk *explains itself* and boundary facts exist in two chunks via overlap.

**✏️ Your turn:** break strategy B — edit `POLICY_DOC` to merge two sections into one giant paragraph (as real docs do) and re-run. Watch B fall while C holds.

---
## Experiment 5 — Where vectors lose 📉

The honest test. Exact identifiers are BM25's home turf:

In [ ]:
tricky = [
    "X-500",                        # bare model number
    "P-113",                        # bare product ID
    "extension board 6-socket",     # exact-ish spec
]
for q in tricky:
    b = retrieve_bm25(q, k=1)[0]
    e = retrieve_semantic(q, k=1)[0]
    b_ok = "✅" if chunk_id(b[0]) in ("P-116", "P-113") else "❌"
    e_ok = "✅" if chunk_id(e[0]) in ("P-116", "P-113") else "❌"
    print(f"Q: {q}")
    print(f"  BM25      {b_ok} → {b[0][:60]}...")
    print(f"  Embeddings {e_ok} → {e[0][:60]}...")
    print()

**Rare tokens like "X-500" have weak, generic embeddings** — the vector barely knows it's a scanner. BM25 matches the string exactly and wins. Complementary failure modes:

| | BM25 | Embeddings |
|---|---|---|
| Model numbers, IDs, exact specs | ✅ | ⚠️ |
| Synonyms, paraphrases, Hinglish | ❌ | ✅ |

**Tomorrow:** hybrid search — run both, merge the rankings — inside ChromaDB with an HNSW index doing the fast neighbour lookup.

---
## 🏁 Finale — RAGBot v0.2

In [ ]:
ALL_CHUNKS = CATALOG + chunk_context_aware(POLICY_DOC)
all_vecs = embed_batch(ALL_CHUNKS)
all_vecs = all_vecs / np.linalg.norm(all_vecs, axis=1, keepdims=True)

RAG_TEMPLATE = """You are SmartBot, the customer assistant of SmartMart retail store, Pune.

Answer ONLY from the context below. If the answer is not in the context,
say "I don't have that information — let me connect you with our team."
Cite the product/policy ID or section you used.
Answer in max 3 sentences, warm and professional.

CONTEXT:
{context}

CUSTOMER QUESTION: {question}"""

class RAGBotV2:
    """v0.2 — semantic retrieval over catalog + context-aware chunked policies."""

    def __init__(self, k=3):
        self.k = k

    def retrieve(self, query):
        q = embed_batch([query])[0]; q = q / np.linalg.norm(q)
        idx = np.argsort(all_vecs @ q)[::-1][:self.k]
        return [ALL_CHUNKS[i] for i in idx]

    def say(self, question, verbose=False):
        chunks = self.retrieve(question)
        if verbose:
            for c in chunks: print(f"   ⤷ {c[:75]}...")
        return ask(RAG_TEMPLATE.format(context="\n".join(chunks), question=question))

bot = RAGBotV2(k=3)
for q in ["sasta kettle dikhao",
          "I lost my receipt, can I still return my fan?",
          "do premium members pay for delivery on small orders?",
          "can I pay for the robot vacuum in installments?"]:
    print(f"🧑 {q}")
    print(f"🛒 {bot.say(q)}")
    print("-" * 70)

In [ ]:
# The v0.1 vs v0.2 scorecard — your deliverable line for the mini-project:
print("=========== RAGBot SCORECARD (k=3) ===========")
print(f"{'metric':<8} {'v0.1 BM25':>12} {'v0.2 semantic':>15}")
for m in ["P@k", "R@k", "F1", "MRR"]:
    print(f"{m:<8} {bm[m]:>12.2f} {em[m]:>15.2f}")
print()
print(f"Policy hit-rate@2 with context-aware chunking: {eval_chunking(chunk_context_aware(POLICY_DOC)):.0%}")

---
## 🏆 Homework (15 min, feeds the Week 2 mini-project)

1. **Chunk your own document** — take any real document you have (college rules PDF, hostel policy, a product manual — copy the text). Apply `chunk_context_aware` (adapt the section regex if needed), write a 5-query mini gold set, and report the hit-rate. This is the exact skill the mini-project grades.
2. **Beat the X-500 failure** — without changing the retriever, fix the "X-500" query by *changing the chunk text* (hint: rule 3 — what alias/context could you inject into P-116's chunk?). Prove it with a re-run.
3. **Threshold thinking** — the retriever always returns k chunks, even for "do you sell helicopters?" Print the top-1 cosine score for 3 sensible and 3 nonsense queries. Propose a score threshold below which the bot should skip retrieval and say "we don't carry that." (This becomes a guardrail in Week 4!)

### What's next
**Module 3 — Vector stores with ChromaDB:** stop hand-rolling numpy — a real vector database with persistence, metadata filters, and an **HNSW** index that searches millions of vectors in milliseconds. Plus **hybrid search**: BM25 + embeddings, merged.

---
*PMS RoBoTics Research Center · pmsroboticsrc@gmail.com · (+91) 9960664674*